In [5]:
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score

In [14]:
db = pd.read_csv('Project_DB_loan_approval.csv')
db = db[db['person_age'] > 18]
train_db = db.copy()
train_db = train_db.drop(['person_age','person_gender','person_education','person_home_ownership','loan_intent',
                          'cb_person_cred_hist_length'], axis = 1)
train_db['previous_loan_defaults_on_file'] = train_db['previous_loan_defaults_on_file'].map({'No' : 0 , 'Yes' : 1})

y_label = train_db['loan_status']
x_features = train_db.copy().drop(['loan_status'], axis = 1)
x_train, x_test, y_train, y_test = train_test_split(x_features, y_label, test_size=0.2, random_state=42, stratify=y_label)
scaler = StandardScaler()

def find_best_svc_pipeline(x_train, y_train, C_range, cv=5):
    """
    Performs a hyperparameter search over C for an SVC (kernel='rbf') 
    using Cross-Validation, and returns the fitted Pipeline with the 
    best-performing C value.

    Args:
        x_train - the normalizetion is happening inside the pipline
        y_train - the correct labels
        C_range - checking C for each point
        cv - the number of "splits" of the data inside x_train to make the evaluation more relaible

    Returns:
        tuple: best_pipeline: Pipeline of x_train
                best_C: value C
                best_score: the "winning" mean Accuracy
    """
    best_score = 0
    best_C = None
    best_pipeline = None

    for c_val in C_range:
        pipe = Pipeline([
            ('scaler', StandardScaler()),
            ('model', SVC(kernel='rbf', C=c_val))
        ])
        scores = cross_val_score(pipe, x_train, y_train, cv=cv, scoring='accuracy')
        mean_score = np.mean(scores)
        print(f"Testing C={c_val} | Mean Accuracy: {mean_score:.4f}")

        if mean_score > best_score:
            best_score = mean_score
            best_C = c_val
            best_pipeline = pipe
            
    print("-" * 40)
    print(f"The winning parameter is C={best_C} with an accuracy of {best_score:.4f}")
    best_pipeline.fit(x_train, y_train)
    y_pred_best = best_pipeline.predict(x_test)
    cm = confusion_matrix(y_test,y_pred_best)
    print("\nAccuracy:\n", accuracy_score(y_test, y_pred_best))
    print("Confusion matrix:\n", cm)
    print("Classification Report:\n")
    print(classification_report(y_test, y_pred_best))
    return best_pipeline, best_C, best_score
best_pipeline, best_C, best_score = find_best_svc_pipeline(x_train, y_train, C_range=[0.1, 1, 10])

Testing C=0.1 | Mean Accuracy: 0.8976
Testing C=1 | Mean Accuracy: 0.9018
Testing C=10 | Mean Accuracy: 0.9029
----------------------------------------
The winning parameter is C=10 with an accuracy of 0.9029

Accuracy:
 0.9044444444444445
Confusion matrix:
 [[6649  351]
 [ 509 1491]]
Classification Report:

              precision    recall  f1-score   support

           0       0.93      0.95      0.94      7000
           1       0.81      0.75      0.78      2000

    accuracy                           0.90      9000
   macro avg       0.87      0.85      0.86      9000
weighted avg       0.90      0.90      0.90      9000



In [15]:
import joblib
joblib.dump(best_pipeline, 'Full project.pkl')
print("File saved!")

File saved!
